# Exploratory Data Analysis: NCAA Lacrosse Team Stats

Phase 2 of the Lacrosse Analytics project. We explore the model-ready dataset to understand distributions, correlations with winning, missing data patterns, and differences between D1 and D2.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

DATA_PATH = Path("../data/processed/team_stats_model_ready.csv")
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"Years: {sorted(df.academic_year.unique())}")
df.head()

## 1. Target Distribution: winning_percentage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["winning_percentage"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Winning Percentage")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Winning Percentage")

for div in [1, 2]:
    subset = df[df["division"] == div]["winning_percentage"]
    axes[1].hist(subset, bins=25, alpha=0.6, label=f"D{div}")
axes[1].set_xlabel("Winning Percentage")
axes[1].set_ylabel("Count")
axes[1].set_title("By Division")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean: {df['winning_percentage'].mean():.3f}")
print(f"Std:  {df['winning_percentage'].std():.3f}")
print(f"Min:  {df['winning_percentage'].min():.3f}, Max: {df['winning_percentage'].max():.3f}")

## 2. Feature Distributions & Outliers

In [ ]:
STAT_COLS = [
    "assists_per_game", "caused_turnovers_per_game", "clearing_percentage",
    "face_off_winning_percentage", "ground_balls_per_game", "man_down_defense",
    "man_up_offense", "opponent_clear_percentage", "points_per_game", "saves_per_game",
    "scoring_defense", "scoring_margin", "scoring_offense", "shot_percentage",
    "turnovers_per_game",
]

# Summary stats
df[STAT_COLS].describe()

In [ ]:
# Box plots for key stats (subset)
key_stats = ["scoring_offense", "scoring_defense", "face_off_winning_percentage", "clearing_percentage"]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flat, key_stats):
    df.boxplot(column=col, by="division", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("Division")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 3. Correlation with Winning Percentage

In [ ]:
# Correlations with target
available = [c for c in STAT_COLS if c in df.columns and df[c].notna().sum() > 100]
corrs = df[available + ["winning_percentage"]].corr()["winning_percentage"].drop("winning_percentage", errors="ignore")
corrs = corrs.sort_values(key=abs, ascending=False)
print("Correlation with winning_percentage:")
print(corrs.to_string())

In [ ]:
# Heatmap of feature correlations
corr_matrix = df[available + ["winning_percentage"]].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 4. Missing Data Analysis

In [ ]:
missing = df[STAT_COLS].isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
print("Missing values per stat:")
for col in STAT_COLS:
    print(f"  {col}: {missing[col]} ({missing_pct[col]}%)")

complete = df[STAT_COLS].dropna(how="any")
print(f"\nRows with ALL stats present: {len(complete)} / {len(df)} ({100*len(complete)/len(df):.1f}%)")

In [ ]:
# opponent_clear_percentage: only available from 2020+
opp_years = df.groupby("academic_year")["opponent_clear_percentage"].apply(lambda x: x.notna().sum())
print("opponent_clear_percentage availability by year:")
print(opp_years.to_string())

## 5. D1 vs D2 Comparison

In [ ]:
print("Sample size by division:")
print(df.groupby("division").size())

print("\nMean winning_percentage by division:")
print(df.groupby("division")["winning_percentage"].mean())

print("\nMean scoring_offense by division:")
print(df.groupby("division")["scoring_offense"].mean())

## 6. COVID Years (2020-2021)

In [ ]:
import re

def total_games(record):
    if pd.isna(record) or not isinstance(record, str):
        return None
    m = re.match(r"^(\d+)-(\d+)$", str(record).strip())
    return int(m.group(1)) + int(m.group(2)) if m else None

df["total_games"] = df["record"].apply(total_games)

print("Average total games per season (W-L):")
print(df.groupby("academic_year")["total_games"].mean().round(1).to_string())
print("\n2020 and 2021 have shortened seasons; consider flagging or excluding for modeling.")

## 7. Year-over-Year Trends

In [ ]:
year_means = df.groupby("academic_year")[["scoring_offense", "scoring_defense", "winning_percentage"]].mean()
year_means.plot(figsize=(10, 4), marker="o")
plt.title("Year-over-Year League Averages")
plt.xlabel("Academic Year")
plt.ylabel("Mean")
plt.legend(bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Summary for modeling decisions
print("EDA Summary:")
print("- Target: winning_percentage, roughly symmetric, range 0-1")
print("- Most stats ~97%+ complete; opponent_clear_pct ~43% complete (2020+ only)")
print("- scoring_margin ~76% complete")
print("- Consider: drop opponent_clear_pct or use subset; impute scoring_margin")
print("- COVID years: 2020 (~6 games), 2021 (~12 games) vs ~15 for others")